# 0. Setup

In [1]:
import ibis
import pandas as pd
from utils.f_0_dirs import get_data_dirs
from linearmodels.panel.results import PanelEffectsResults
from f_7_run_panel import run_panel, ModelSpec, format_str, reindex_entity

dirs = get_data_dirs(segment="model")
con = ibis.duckdb.connect(dirs.db_path, read_only=True)

# 0. 2-factor productivity

In [ ]:
out_file_name = "results_0_tfp"

table_panel_name = "working_yearly"
table_panel_old = "fame_yearly_kp"

t_panel = con.table(table_panel_name)
t_old = con.table(table_panel_old)
df_panel = (
    t_panel
    .drop('gva1_per_worker', 'gva2_per_worker')
    .distinct(on=['registered_number', 'year'])
    .left_join(
        t_old.select('registered_number', 'year', 'tangibles', 'intangibles', 'investments_other').distinct(on=['registered_number', 'year']),
        ['registered_number', 'year']
    )
    .execute()
)

models = {
    'gva1_ft': {'Y': 'gva1', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['i', 't']},
    'gva1_f': {'Y': 'gva1', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['i']},
    'gva1_t': {'Y': 'gva1', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['t']},
    'gva1_n': {'Y': 'gva1', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': []}
}

run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args = [
    (ModelSpec(**mod_obj), df_panel, m_name)
    for m_name, mod_obj in models.items() if ModelSpec(**mod_obj).include
]
for args in worker_args:
    mod, table_panel_name, model_name = args
    res, beta, effects_dict = run_panel(args)
    if res is None:
        print(f"Model '{model_name}' failed. Skipping.")
        continue
    run_res_series.append((res, mod, model_name))

model_count = len(run_res_series)
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing.")
    with open(dirs.output_dir / f"{out_file_name}.txt", "w") as f:
        f.write(output_str)

Running model 'gva1_ft'
✅ Model 'gva1_ft' estimated: ln_total_assets=0.286, ln_employees=0.617
Running model 'gva1_f'
✅ Model 'gva1_f' estimated: ln_total_assets=0.284, ln_employees=0.613
Running model 'gva1_t'
✅ Model 'gva1_t' estimated: ln_total_assets=0.421, ln_employees=0.562
Running model 'gva1_n'
✅ Model 'gva1_n' estimated: ln_total_assets=0.421, ln_employees=0.561
Panel regressions complete. 4 models, writing.


In [ ]:
out_file_name = "results_0_tfp_alt_measure"

models = {
    'gva1': {'Y': 'gva1', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['i', 't']},
    'gva2': {'Y': 'gva2', 'X': ['total_assets', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['i', 't']},
    'k1': {'Y': 'gva1', 'X': ['fixed_total', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['i', 't']},
    'k2': {'Y': 'gva1', 'X': ['tangibles', 'intangibles', 'investments_other', 'employees'], 'to_log': ['Y', 'X'], 'fe': ['i', 't']}
}

run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args = [
    (ModelSpec(**mod_obj), df_panel, m_name)
    for m_name, mod_obj in models.items() if ModelSpec(**mod_obj).include
]
for args in worker_args:
    mod, table_panel_name, model_name = args
    res, beta, effects_dict = run_panel(args)
    if res is None:
        print(f"Model '{model_name}' failed. Skipping.")
        continue
    run_res_series.append((res, mod, model_name))

model_count = len(run_res_series)
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing.")
    with open(dirs.output_dir / f"{out_file_name}.txt", "w") as f:
        f.write(output_str)

Running model 'gva1'
✅ Model 'gva1' estimated: ln_total_assets=0.286, ln_employees=0.617
Running model 'gva2'
✅ Model 'gva2' estimated: ln_total_assets=0.265, ln_employees=0.611
Running model 'gva3'
❌ Model 'gva3' failed. KeyError: 'gva3'
Model 'gva3' failed. Skipping.
Running model 'k1'


Traceback (most recent call last):
  File "c:\Users\lazyst\AppData\Local\python\pythoncore-3.14-64\Lib\site-packages\pandas\core\indexes\base.py", line 3641, in get_loc
    return self._engine.get_loc(casted_key)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 168, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 197, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7668, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7676, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'gva3'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\lazyst\AppData\Local\Temp\ipykernel_28800\996074865.py", line 60, in run_panel
    f'ln_{p}': df_raw[p].apply(lambda x: np.log(x) if x > 0 else None) for p in params_named if p in mod.to_log
              

✅ Model 'k1' estimated: ln_fixed_total=0.065, ln_employees=0.720
Running model 'k2'
✅ Model 'k2' estimated: ln_tangibles=0.090, ln_intangibles=0.025, ln_investments_other=0.007, ln_employees=0.628
Panel regressions complete. 4 models, writing.


# 1a. LMM
$$
\begin{align*}
y_{it} &= \alpha_i + \gamma_t + w_{it}\theta + \beta E[TFP_{-i,g,t}\vert{}g] + \epsilon_{it} \\
x_{it} &=
\begin{pmatrix}
k_{it} & l_{it}
\end{pmatrix}
\end{align*}
$$
**Run 1**
- ✅ Model 'peer3' estimated: peer_tfp_ttwa_donut=0.033, peer_tfp_pc4_donut=0.015, peer_tfp_pc8=0.280
- ✅ Model 'peer3_no_firm_fe' estimated: peer_tfp_ttwa_donut=0.204, peer_tfp_pc4_donut=0.142, peer_tfp_pc8=0.559
- ✅ Model 'peer3_employees' estimated: ln_employees=0.009, peer_tfp_ttwa_donut=0.033, peer_tfp_pc4_donut=0.015, peer_tfp_pc8=0.280
Panel regressions complete. 3 models, writing.

In [ ]:
panel_name = "working_yearly_g"
out_name = "results_1a_lmm_exog"

# t_diffed = ibis.read_parquet(dirs.output_dir / f"{panel_name}_diff.parquet", engine="pyarrow")
df_diff = pd.read_parquet(dirs.tmp_dir / f"{panel_name}_diff.parquet", engine="pyarrow")
# df_balanced: pd.DataFrame = (
#     con.table(panel_name)
#     .distinct(on=['registered_number', 'year'])
#     .execute()          # type: ignore
#     .set_index(['registered_number', 'year'])
#     .sort_index()
#     .groupby(level='registered_number', group_keys=False)
#     .apply(reindex_entity)
# )
# df_diff = df_balanced.copy()
# for col in df_balanced.columns:
#     df_diff[col] = df_balanced.groupby(level='registered_number')[col].diff(1)
print(df_diff.head())
df_diff['registered_number'] = df_diff.index.get_level_values('registered_number')
df_diff['year'] = df_diff.index.get_level_values('year')
# print(df_diff.head())

# 3. Models: varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
models = {
    'lmm_exog': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['t'],
        'description': 'Strict exogeneity, structural form'
    },
    'lmm_instr': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'Z': {
            'wg1_y': ['w2g1_k', 'w2g1_l'],
        },
        'fe': ['t'],
        'description': 'Instrument endogenous effect, LMM',
        # 'use_linearmodels': False,
        # 'differencing': 'mean'
    }
}

run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args = [
    (ModelSpec(**mod_obj), panel_name, m_name)
    for m_name, mod_obj in models.items() if ModelSpec(**mod_obj).include
]

for args in worker_args:
    mod, _, model_name = args
    res, beta, effects_dict = run_panel((mod, df_diff, model_name))
    if res is None:
        print(f"Model '{model_name}' failed. Skipping.")
        continue
    run_res_series.append((res, mod, model_name))   # type: ignore

model_count = len(run_res_series)
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing to {out_name}.txt")
    with open(dirs.output_dir / f"{out_name}.txt", "w") as f:
        f.write(output_str)

                                 gva   total_assets  employees         y  \
registered_number year                                                     
00000006          2012           NaN            NaN        NaN       NaN   
                  2013 -47355.547044 -571047.150314      -24.0 -2.249008   
00000140          2006           NaN            NaN        NaN       NaN   
                  2007    886.356968    3012.891278        4.0  0.071164   
                  2008  -1052.096783   -1448.867717        2.0 -0.085052   

                               k         l     wg1_y     wg2_y     wg3_y  \
registered_number year                                                     
00000006          2012       NaN       NaN       NaN       NaN       NaN   
                  2013 -0.153467 -0.093819 -0.115470  0.002254  0.025306   
00000140          2006       NaN       NaN       NaN       NaN       NaN   
                  2007  0.102214  0.012270 -0.072460 -0.502603  0.056936   
           

In [8]:
df_diff.to_parquet(dirs.output_dir / f"{panel_name}_diff.parquet", engine="pyarrow", index=True)

In [ ]:
# for args in worker_args:
#     mod, _, model_name = args
#     res, beta, effects_dict = run_panel(args)
#     if res is None:
#         print(f"Model '{model_name}' failed. Skipping.")
#         continue
#     run_res_series.append((res, mod, model_name))   # type: ignore

# model_count = len(run_res_series)
# if model_count:
#     output_str = "\n".join(map(format_str, run_res_series))
#     print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing.")
#     with open(dirs.output_dir / f"{out_name}.txt", "w") as f:
#         f.write(output_str)

# 2. Distance decay model

$$
\begin{align*}
z_{it} &= \alpha_i + \gamma_t + \rho \sum_{j \neq i} f(d_{ij}) \cdot z_{jt} + \epsilon_{it}   \\
y_{it} &= \alpha_i + \gamma_t + \beta_1 k_{it} + \beta_2 l_{it} + \rho \sum_{j \neq i} w_{ij} z_{jt} + \epsilon_{it}
\end{align*}
$$

In [3]:
from linearmodels.panel.results import PanelEffectsResults
from f_7_run_panel import run_panel, ModelSpec, format_str

panel_name = "working_yearly_n"
out_name = "results_2a_dd"

df_diff = pd.read_parquet(dirs.tmp_dir / f"{panel_name}_diff.parquet", engine="pyarrow")
df_diff['registered_number'] = df_diff.index.get_level_values('registered_number')
df_diff['year'] = df_diff.index.get_level_values('year')
print(df_diff.head())

# 3. Models: varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
m_models = {
    'dd1': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['i', 't'],
        'description': 'Distance 1 model'
    },
    'dd2': {
        'Y': 'y',
        'X': ['k', 'l', 'wd2_y', 'wd2_k', 'wd2_l'],
        'Z': {
            'wd2_y': ['w2d2_k', 'w2d2_l']
        },
        'fe': ['i', 't'],
        'description': 'Distance 2 model'
    },
    'dd3': {
        'Y': 'y',
        'X': ['k', 'l', 'wd3_y', 'wd3_k', 'wd3_l'],
        'Z': {
            'wd3_y': ['w2d3_k', 'w2d3_l']
        },
        'fe': ['i', 't'],
        'description': 'Distance 3 model'
    }
}
iv_models = {
    'dd1-2lag': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['i', 't'],
        'description': '2nd-order spatial lag model'
    },
    'dd1-3lag': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l', 'w3d1_k', 'w3d1_l']
        },
        'fe': ['i', 't'],
        'description': '3rd-order spatial lag model'
    }
}

models = m_models

run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args = [
    (ModelSpec(**mod_obj), panel_name, m_name)
    for m_name, mod_obj in models.items() if ModelSpec(**mod_obj).include
]

for args in worker_args:
    mod, _, model_name = args
    res, beta, effects_dict = run_panel((mod, df_diff, model_name))
    if res is None:
        print(f"Model '{model_name}' failed. Skipping.")
        continue
    run_res_series.append((res, mod, model_name))   # type: ignore

model_count = len(run_res_series)
if model_count:
    output_str = "\n".join(map(format_str, run_res_series))
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}, writing to {out_name}.txt")
    with open(dirs.output_dir / f"{out_name}.txt", "w") as f:
        f.write(output_str)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\lazyst\\Files\\ucl\\Dissertation\\model\\tmp\\working_yearly_n_diff.parquet'